In [ ]:
import pandas as pd
df = pd.read_parquet('data/processed_data/combined_subjects.parquet')
df = df[df['sub'] == 3]
df['CO2'] = df['CO2'] - df['CO2'].shift(1)
#dropna in co2 column
df = df.dropna(subset=['CO2'])


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# --- 2. Calculate the Sample Covariogram ---
# The covariogram C(h) is calculated for discrete lags h. We create a 2D grid of lags.

# Parameters for the lag bins
n_lags = 10  # Number of lag bins in each direction (x and y) from the center
# Use the maximum distance in the data to define the lag limits
max_lag_x = (df['x'].max() - df['x'].min()) / 2
max_lag_y = (df['y'].max() - df['y'].min()) / 2

# Define the lag bin edges. The number of edges is (n_lags * 2) + 1.
# This creates a grid of bins centered at (0,0).
lag_x_bins = np.linspace(-max_lag_x, max_lag_x, n_lags * 2 + 1)
lag_y_bins = np.linspace(-max_lag_y, max_lag_y, n_lags * 2 + 1)
n_bins_x = len(lag_x_bins) - 1
n_bins_y = len(lag_y_bins) - 1

# Initialize grids to store the sum of covariance products and the number of pairs in each bin
covariogram_sum = np.zeros((n_bins_x, n_bins_y))
pair_count = np.zeros((n_bins_x, n_bins_y))

# Get data as numpy arrays for faster computation
coords = df[['x', 'y']].values
values = df['CO2'].values
mean_co2 = values.mean()

# Iterate over all unique pairs of points
for i in range(len(df)):
    for j in range(i, len(df)):
        # Calculate the lag vector h = s_i - s_j
        lag_vector = coords[i] - coords[j]
        h_x, h_y = lag_vector[0], lag_vector[1]

        # Calculate the covariance product for this pair
        cov_product = (values[i] - mean_co2) * (values[j] - mean_co2)

        # Find the appropriate bin index for this lag vector (h_x, h_y)
        # np.digitize returns an index from 1 to len(bins). We subtract 1 for 0-based indexing.
        bin_x = np.digitize(h_x, lag_x_bins) - 1
        bin_y = np.digitize(h_y, lag_y_bins) - 1

        # Add the product to the corresponding bin if it's within our grid
        if 0 <= bin_x < n_bins_x and 0 <= bin_y < n_bins_y:
            covariogram_sum[bin_x, bin_y] += cov_product
            pair_count[bin_x, bin_y] += 1
            # Since C(h) = C(-h), we add the same value for the opposite lag vector
            if i != j:
                neg_bin_x = np.digitize(-h_x, lag_x_bins) - 1
                neg_bin_y = np.digitize(-h_y, lag_y_bins) - 1
                if 0 <= neg_bin_x < n_bins_x and 0 <= neg_bin_y < n_bins_y:
                    covariogram_sum[neg_bin_x, neg_bin_y] += cov_product
                    pair_count[neg_bin_x, neg_bin_y] += 1

# Calculate the average covariance for each bin
# Handle bins with no pairs (division by zero) by leaving them as zero.
sample_covariogram = np.divide(covariogram_sum, pair_count, out=np.zeros_like(covariogram_sum), where=pair_count!=0)

# --- 3. Plot the 3D Sample Covariogram ---

# Get the center points of each bin for plotting coordinates
lag_x_centers = (lag_x_bins[:-1] + lag_x_bins[1:]) / 2
lag_y_centers = (lag_y_bins[:-1] + lag_y_bins[1:]) / 2
X, Y = np.meshgrid(lag_x_centers, lag_y_centers)

# The Z values are the calculated covariogram values. We need to transpose them to match the meshgrid's orientation.
Z = sample_covariogram.T

fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

# Plot the surface
surf = ax.plot_surface(X, Y, Z, cmap='viridis', edgecolor='none')

# Add labels and title
ax.set_title('3D Sample Covariogram of CO2 Data', fontsize=16)
ax.set_xlabel('Lag in x-direction ($h_x$)', fontsize=12)
ax.set_ylabel('Lag in y-direction ($h_y$)', fontsize=12)
ax.set_zlabel('Covariance $C(\mathbf{h})$', fontsize=12)

# Adjust viewing angle for better perspective
ax.view_init(elev=45, azim=-65)

# Add a color bar which maps values to colors
fig.colorbar(surf, shrink=0.6, aspect=10, label='Covariance')

plt.show()



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# --- 2. Calculate the 2D Sample Covariogram (for 3D plot) ---
# The covariogram C(h) is calculated for discrete lags h. We create a 2D grid of lags.

# Parameters for the lag bins
n_lags_2d = 10  # Number of lag bins in each direction (x and y) from the center
max_lag_x = (df['x'].max() - df['x'].min()) / 2
max_lag_y = (df['y'].max() - df['y'].min()) / 2

# Define the lag bin edges for the 2D grid
lag_x_bins = np.linspace(-max_lag_x, max_lag_x, n_lags_2d * 2 + 1)
lag_y_bins = np.linspace(-max_lag_y, max_lag_y, n_lags_2d * 2 + 1)
n_bins_x = len(lag_x_bins) - 1
n_bins_y = len(lag_y_bins) - 1

# Initialize grids to store the sum of covariance products and the number of pairs in each bin
covariogram_sum_2d = np.zeros((n_bins_x, n_bins_y))
pair_count_2d = np.zeros((n_bins_x, n_bins_y))

# Get data as numpy arrays for faster computation
coords = df[['x', 'y']].values
values = df['CO2'].values
mean_co2 = values.mean()

# Iterate over all unique pairs of points for the 2D covariogram
for i in range(len(df)):
    for j in range(i, len(df)):
        lag_vector = coords[i] - coords[j]
        h_x, h_y = lag_vector[0], lag_vector[1]
        cov_product = (values[i] - mean_co2) * (values[j] - mean_co2)
        bin_x = np.digitize(h_x, lag_x_bins) - 1
        bin_y = np.digitize(h_y, lag_y_bins) - 1

        if 0 <= bin_x < n_bins_x and 0 <= bin_y < n_bins_y:
            covariogram_sum_2d[bin_x, bin_y] += cov_product
            pair_count_2d[bin_x, bin_y] += 1
            if i != j:
                neg_bin_x = np.digitize(-h_x, lag_x_bins) - 1
                neg_bin_y = np.digitize(-h_y, lag_y_bins) - 1
                if 0 <= neg_bin_x < n_bins_x and 0 <= neg_bin_y < n_bins_y:
                    covariogram_sum_2d[neg_bin_x, neg_bin_y] += cov_product
                    pair_count_2d[neg_bin_x, neg_bin_y] += 1

sample_covariogram_2d = np.divide(covariogram_sum_2d, pair_count_2d, out=np.zeros_like(covariogram_sum_2d), where=pair_count_2d!=0)

# --- 3. Plot the 3D Sample Covariogram ---
lag_x_centers = (lag_x_bins[:-1] + lag_x_bins[1:]) / 2
lag_y_centers = (lag_y_bins[:-1] + lag_y_bins[1:]) / 2
X, Y = np.meshgrid(lag_x_centers, lag_y_centers)
Z = sample_covariogram_2d.T

fig = plt.figure(figsize=(18, 8))
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
surf = ax1.plot_surface(X, Y, Z, cmap='viridis', edgecolor='none')
ax1.set_title('3D Sample Covariogram (Anisotropy Check)', fontsize=16)
ax1.set_xlabel('Lag in x-direction ($h_x$)')
ax1.set_ylabel('Lag in y-direction ($h_y$)')
ax1.set_zlabel('Covariance $C(\mathbf{h})$')
ax1.view_init(elev=45, azim=-65)
fig.colorbar(surf, ax=ax1, shrink=0.6, aspect=10, label='Covariance')


# --- 4. Calculate and Plot Directional Covariograms for Isotropy Check ---
n_lags_1d = 15  # Number of lag bins for 1D plots
max_lag = np.sqrt(max_lag_x**2 + max_lag_y**2)
lag_bins_1d = np.linspace(0, max_lag, n_lags_1d + 1)
lag_centers_1d = (lag_bins_1d[:-1] + lag_bins_1d[1:]) / 2

# Initialize arrays for omnidirectional and directional covariograms
cov_omni = np.zeros(n_lags_1d)
count_omni = np.zeros(n_lags_1d)
# Directions: 0/180 (East-West), 90/270 (North-South), 45/225, 135/315
directions = {
    '0-180': {'cov': np.zeros(n_lags_1d), 'count': np.zeros(n_lags_1d)},
    '90-270': {'cov': np.zeros(n_lags_1d), 'count': np.zeros(n_lags_1d)},
    '45-225': {'cov': np.zeros(n_lags_1d), 'count': np.zeros(n_lags_1d)},
    '135-315': {'cov': np.zeros(n_lags_1d), 'count': np.zeros(n_lags_1d)}
}
angle_tolerance = 10  # Tolerance in degrees to assign a pair to a direction

for i in range(len(df)):
    for j in range(i, len(df)):
        lag_vector = coords[i] - coords[j]
        h_x, h_y = lag_vector[0], lag_vector[1]
        dist = np.sqrt(h_x**2 + h_y**2)

        if dist == 0: continue

        cov_product = (values[i] - mean_co2) * (values[j] - mean_co2)
        bin_1d = np.digitize(dist, lag_bins_1d) - 1

        if 0 <= bin_1d < n_lags_1d:
            # Omnidirectional
            cov_omni[bin_1d] += cov_product
            count_omni[bin_1d] += 1

            # Directional
            angle = np.degrees(np.arctan2(h_y, h_x)) % 360

            # 0/180 direction (E-W)
            if (abs(angle - 0) < angle_tolerance) or (abs(angle - 180) < angle_tolerance) or (abs(angle-360) < angle_tolerance):
                directions['0-180']['cov'][bin_1d] += cov_product
                directions['0-180']['count'][bin_1d] += 1

            # 90/270 direction (N-S)
            if (abs(angle - 90) < angle_tolerance) or (abs(angle - 270) < angle_tolerance):
                directions['90-270']['cov'][bin_1d] += cov_product
                directions['90-270']['count'][bin_1d] += 1

            # 45/225 direction (NE-SW)
            if (abs(angle - 45) < angle_tolerance) or (abs(angle - 225) < angle_tolerance):
                directions['45-225']['cov'][bin_1d] += cov_product
                directions['45-225']['count'][bin_1d] += 1

            # 135/315 direction (NW-SE)
            if (abs(angle - 135) < angle_tolerance) or (abs(angle - 315) < angle_tolerance):
                directions['135-315']['cov'][bin_1d] += cov_product
                directions['135-315']['count'][bin_1d] += 1

# Calculate final covariogram values
final_cov_omni = np.divide(cov_omni, count_omni, out=np.full(n_lags_1d, np.nan), where=count_omni!=0)
for d in directions:
    directions[d]['final_cov'] = np.divide(directions[d]['cov'], directions[d]['count'], out=np.full(n_lags_1d, np.nan), where=directions[d]['count']!=0)

# Plotting
ax2 = fig.add_subplot(1, 2, 2)
ax2.plot(lag_centers_1d, final_cov_omni, 'ko-', label='Omnidirectional', lw=3)
ax2.plot(lag_centers_1d, directions['0-180']['final_cov'], 'r.:', label='0-180 deg (E-W)')
ax2.plot(lag_centers_1d, directions['90-270']['final_cov'], 'b.:', label='90-270 deg (N-S)')
ax2.plot(lag_centers_1d, directions['45-225']['final_cov'], 'g.:', label='45-225 deg (NE-SW)')
ax2.plot(lag_centers_1d, directions['135-315']['final_cov'], 'm.:', label='135-315 deg (NW-SE)')
ax2.set_title('Directional vs. Omnidirectional Covariograms', fontsize=16)
ax2.set_xlabel('Lag Distance $h$', fontsize=12)
ax2.set_ylabel('Covariance $C(h)$', fontsize=12)
ax2.legend()
ax2.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()


In [ ]:
# --- 4. Calculate and Plot Directional Covariograms for Isotropy Check ---
n_lags_1d = 15  # Number of lag bins for 1D plots
max_lag = np.sqrt(max_lag_x**2 + max_lag_y**2)
lag_bins_1d = np.linspace(0, max_lag, n_lags_1d + 1)
lag_centers_1d = (lag_bins_1d[:-1] + lag_bins_1d[1:]) / 2

# Initialize arrays for omnidirectional and directional covariograms
cov_omni = np.zeros(n_lags_1d)
count_omni = np.zeros(n_lags_1d)
# Directions: 0/180 (East-West), 90/270 (North-South), 45/225, 135/315
directions = {
    '0-180': {'cov': np.zeros(n_lags_1d), 'count': np.zeros(n_lags_1d)},
    '90-270': {'cov': np.zeros(n_lags_1d), 'count': np.zeros(n_lags_1d)},
    '45-225': {'cov': np.zeros(n_lags_1d), 'count': np.zeros(n_lags_1d)},
    '135-315': {'cov': np.zeros(n_lags_1d), 'count': np.zeros(n_lags_1d)}
}
angle_tolerance = 22.5  # Tolerance in degrees to assign a pair to a direction

for i in range(len(df)):
    for j in range(i, len(df)):
        lag_vector = coords[i] - coords[j]
        h_x, h_y = lag_vector[0], lag_vector[1]
        dist = np.sqrt(h_x**2 + h_y**2)

        if dist == 0: continue

        cov_product = (values[i] - mean_co2) * (values[j] - mean_co2)
        bin_1d = np.digitize(dist, lag_bins_1d) - 1

        if 0 <= bin_1d < n_lags_1d:
            # Omnidirectional
            cov_omni[bin_1d] += cov_product
            count_omni[bin_1d] += 1

            # Directional
            angle = np.degrees(np.arctan2(h_y, h_x)) % 360

            # 0/180 direction (E-W)
            if (abs(angle - 0) < angle_tolerance) or (abs(angle - 180) < angle_tolerance) or (abs(angle-360) < angle_tolerance):
                directions['0-180']['cov'][bin_1d] += cov_product
                directions['0-180']['count'][bin_1d] += 1

            # 90/270 direction (N-S)
            if (abs(angle - 90) < angle_tolerance) or (abs(angle - 270) < angle_tolerance):
                directions['90-270']['cov'][bin_1d] += cov_product
                directions['90-270']['count'][bin_1d] += 1

            # 45/225 direction (NE-SW)
            if (abs(angle - 45) < angle_tolerance) or (abs(angle - 225) < angle_tolerance):
                directions['45-225']['cov'][bin_1d] += cov_product
                directions['45-225']['count'][bin_1d] += 1

            # 135/315 direction (NW-SE)
            if (abs(angle - 135) < angle_tolerance) or (abs(angle - 315) < angle_tolerance):
                directions['135-315']['cov'][bin_1d] += cov_product
                directions['135-315']['count'][bin_1d] += 1

# Calculate final covariogram values
final_cov_omni = np.divide(cov_omni, count_omni, out=np.full(n_lags_1d, np.nan), where=count_omni!=0)
for d in directions:
    directions[d]['final_cov'] = np.divide(directions[d]['cov'], directions[d]['count'], out=np.full(n_lags_1d, np.nan), where=directions[d]['count']!=0)

# Plotting
ax2 = fig.add_subplot(1, 2, 2)
ax2.plot(lag_centers_1d, final_cov_omni, 'ko-', label='Omnidirectional', lw=3)
ax2.plot(lag_centers_1d, directions['0-180']['final_cov'], 'r.:', label='0-180 deg (E-W)')
ax2.plot(lag_centers_1d, directions['90-270']['final_cov'], 'b.:', label='90-270 deg (N-S)')
ax2.plot(lag_centers_1d, directions['45-225']['final_cov'], 'g.:', label='45-225 deg (NE-SW)')
ax2.plot(lag_centers_1d, directions['135-315']['final_cov'], 'm.:', label='135-315 deg (NW-SE)')
ax2.set_title('Directional vs. Omnidirectional Covariograms', fontsize=16)
ax2.set_xlabel('Lag Distance $h$', fontsize=12)
ax2.set_ylabel('Covariance $C(h)$', fontsize=12)
ax2.legend()
ax2.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
# The Z values are the calculated covariogram values. We need to transpose them to match the meshgrid's orientation.
Z = sample_covariogram.T

fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

# Plot the surface
surf = ax.plot_surface(X, Y, Z, cmap='viridis', edgecolor='none')

# Add labels and title
ax.set_title('3D Sample Covariogram of CO2 Data', fontsize=16)
ax.set_xlabel('Lag in x-direction ($h_x$)', fontsize=12)
ax.set_ylabel('Lag in y-direction ($h_y$)', fontsize=12)
ax.set_zlabel('Covariance $C(\mathbf{h})$', fontsize=12)

# Adjust viewing angle for better perspective
ax.view_init(elev=0, azim=90)

# Add a color bar which maps values to colors
fig.colorbar(surf, shrink=0.6, aspect=10, label='Covariance')

plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D


# --- 2. Check for Trends (Visual Inspection) ---
fig_trend = plt.figure(figsize=(12, 5))
ax_trend1 = fig_trend.add_subplot(1, 2, 1)
sc1 = ax_trend1.scatter(df['x'], df['CO2'], c=df['CO2'], cmap='viridis')
ax_trend1.set_title('CO2 vs. X-coordinate')
ax_trend1.set_xlabel('X-coordinate')
ax_trend1.set_ylabel('CO2')
fig_trend.colorbar(sc1, ax=ax_trend1, label='CO2')

ax_trend2 = fig_trend.add_subplot(1, 2, 2)
sc2 = ax_trend2.scatter(df['y'], df['CO2'], c=df['CO2'], cmap='viridis')
ax_trend2.set_title('CO2 vs. Y-coordinate')
ax_trend2.set_xlabel('Y-coordinate')
ax_trend2.set_ylabel('CO2')
fig_trend.colorbar(sc2, ax=ax_trend2, label='CO2')

plt.suptitle("Trend Analysis: Check for Constant Mean")
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()


# --- 3. Calculate 2D Variogram for 3D Plot ---
coords = df[['x', 'y']].values
values = df['CO2'].values

n_lags_2d = 10  # Number of lag bins in each direction
max_lag_x = (df['x'].max() - df['x'].min()) / 2
max_lag_y = (df['y'].max() - df['y'].min()) / 2

lag_x_bins = np.linspace(-max_lag_x, max_lag_x, n_lags_2d * 2 + 1)
lag_y_bins = np.linspace(-max_lag_y, max_lag_y, n_lags_2d * 2 + 1)
n_bins_x = len(lag_x_bins) - 1
n_bins_y = len(lag_y_bins) - 1

variogram_sum_2d = np.zeros((n_bins_x, n_bins_y))
pair_count_2d = np.zeros((n_bins_x, n_bins_y))

for i in range(len(df)):
    for j in range(i + 1, len(df)):
        lag_vector = coords[i] - coords[j]
        sq_diff = (values[i] - values[j])**2

        # Process positive lag vector
        h_x, h_y = lag_vector[0], lag_vector[1]
        bin_x = np.digitize(h_x, lag_x_bins) - 1
        bin_y = np.digitize(h_y, lag_y_bins) - 1
        if 0 <= bin_x < n_bins_x and 0 <= bin_y < n_bins_y:
            variogram_sum_2d[bin_x, bin_y] += sq_diff
            pair_count_2d[bin_x, bin_y] += 1

        # Process negative lag vector
        neg_h_x, neg_h_y = -h_x, -h_y
        neg_bin_x = np.digitize(neg_h_x, lag_x_bins) - 1
        neg_bin_y = np.digitize(neg_h_y, lag_y_bins) - 1
        if 0 <= neg_bin_x < n_bins_x and 0 <= neg_bin_y < n_bins_y:
            variogram_sum_2d[neg_bin_x, neg_bin_y] += sq_diff
            pair_count_2d[neg_bin_x, neg_bin_y] += 1

sample_variogram_2d = np.divide(variogram_sum_2d, pair_count_2d, out=np.full_like(variogram_sum_2d, np.nan), where=pair_count_2d > 0)

# --- 4. Calculate Directional Variograms ---
n_lags_1d = 15  # Number of lag bins
max_lag_1d = np.sqrt(max_lag_x**2 + max_lag_y**2)
lag_bins_1d = np.linspace(0, max_lag_1d, n_lags_1d + 1)
lag_centers_1d = (lag_bins_1d[:-1] + lag_bins_1d[1:]) / 2

directions = {
    'Omnidirectional': {'sum_sq_diff': np.zeros(n_lags_1d), 'count': np.zeros(n_lags_1d)},
    '0-180 (E-W)': {'sum_sq_diff': np.zeros(n_lags_1d), 'count': np.zeros(n_lags_1d)},
    '90-270 (N-S)': {'sum_sq_diff': np.zeros(n_lags_1d), 'count': np.zeros(n_lags_1d)},
    '45-225 (NE-SW)': {'sum_sq_diff': np.zeros(n_lags_1d), 'count': np.zeros(n_lags_1d)},
    '135-315 (NW-SE)': {'sum_sq_diff': np.zeros(n_lags_1d), 'count': np.zeros(n_lags_1d)}
}
angle_tolerance = 22.5

for i in range(len(df)):
    for j in range(i + 1, len(df)):
        lag_vector = coords[i] - coords[j]
        h_x, h_y = lag_vector[0], lag_vector[1]
        dist = np.sqrt(h_x**2 + h_y**2)
        bin_index = np.digitize(dist, lag_bins_1d) - 1

        if 0 <= bin_index < n_lags_1d:
            sq_diff = (values[i] - values[j])**2
            directions['Omnidirectional']['sum_sq_diff'][bin_index] += sq_diff
            directions['Omnidirectional']['count'][bin_index] += 1
            angle = np.degrees(np.arctan2(h_y, h_x)) % 180

            if abs(angle - 0) < angle_tolerance or abs(angle - 180) < angle_tolerance: key = '0-180 (E-W)'
            elif abs(angle - 90) < angle_tolerance: key = '90-270 (N-S)'
            elif abs(angle - 45) < angle_tolerance: key = '45-225 (NE-SW)'
            elif abs(angle - 135) < angle_tolerance: key = '135-315 (NW-SE)'
            else: key = None

            if key:
                directions[key]['sum_sq_diff'][bin_index] += sq_diff
                directions[key]['count'][bin_index] += 1

for key in directions:
    count = directions[key]['count']
    directions[key]['variogram'] = np.divide(directions[key]['sum_sq_diff'], count, out=np.full(n_lags_1d, np.nan), where=count > 0)

# --- 5. Plot the Variograms ---
fig = plt.figure(figsize=(18, 8))

# 3D Plot
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
lag_x_centers = (lag_x_bins[:-1] + lag_x_bins[1:]) / 2
lag_y_centers = (lag_y_bins[:-1] + lag_y_bins[1:]) / 2
X, Y = np.meshgrid(lag_x_centers, lag_y_centers)
Z = sample_variogram_2d.T
ax1.plot_surface(X, Y, Z, cmap='viridis', edgecolor='none')
ax1.set_title('3D Sample Variogram', fontsize=16)
ax1.set_xlabel('Lag in x-direction ($h_x$)')
ax1.set_ylabel('Lag in y-direction ($h_y$)')
ax1.set_zlabel('Variogram $2\gamma(\mathbf{h})$')
ax1.view_init(elev=45, azim=-65)

# 2D Directional Plot
ax2 = fig.add_subplot(1, 2, 2)
for key, style in zip(directions.keys(), ['ko-', 'r.:', 'b.:', 'g.:', 'm.:']):
    ax2.plot(lag_centers_1d, directions[key]['variogram'], style, label=f"{key}", lw=3 if key == 'Omnidirectional' else 1)
ax2.set_title('Directional Variograms', fontsize=16)
ax2.set_xlabel('Lag Distance $h$')
ax2.set_ylabel('Variogram $2\gamma(h)$')
ax2.legend()
ax2.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots


# --- 2. Trend analysis is still recommended but plotting is deferred to the final interactive plot ---
# You can visually inspect df.plot(kind='scatter', x='x', y='CO2') etc. if needed.

# --- 3. Calculate 2D Variogram for 3D Plot ---
coords = df[['x', 'y']].values
values = df['CO2'].values

n_lags_2d = 10  # Number of lag bins in each direction
max_lag_x = (df['x'].max() - df['x'].min()) / 2
max_lag_y = (df['y'].max() - df['y'].min()) / 2

lag_x_bins = np.linspace(-max_lag_x, max_lag_x, n_lags_2d * 2 + 1)
lag_y_bins = np.linspace(-max_lag_y, max_lag_y, n_lags_2d * 2 + 1)
n_bins_x = len(lag_x_bins) - 1
n_bins_y = len(lag_y_bins) - 1

variogram_sum_2d = np.zeros((n_bins_x, n_bins_y))
pair_count_2d = np.zeros((n_bins_x, n_bins_y))

for i in range(len(df)):
    for j in range(i + 1, len(df)):
        lag_vector = coords[i] - coords[j]
        sq_diff = (values[i] - values[j])**2

        # Process positive lag vector
        h_x, h_y = lag_vector[0], lag_vector[1]
        bin_x = np.digitize(h_x, lag_x_bins) - 1
        bin_y = np.digitize(h_y, lag_y_bins) - 1
        if 0 <= bin_x < n_bins_x and 0 <= bin_y < n_bins_y:
            variogram_sum_2d[bin_x, bin_y] += sq_diff
            pair_count_2d[bin_x, bin_y] += 1

        # Process negative lag vector
        neg_h_x, neg_h_y = -h_x, -h_y
        neg_bin_x = np.digitize(neg_h_x, lag_x_bins) - 1
        neg_bin_y = np.digitize(neg_h_y, lag_y_bins) - 1
        if 0 <= neg_bin_x < n_bins_x and 0 <= neg_bin_y < n_bins_y:
            variogram_sum_2d[neg_bin_x, neg_bin_y] += sq_diff
            pair_count_2d[neg_bin_x, neg_bin_y] += 1

sample_variogram_2d = np.divide(variogram_sum_2d, pair_count_2d, out=np.full_like(variogram_sum_2d, np.nan), where=pair_count_2d > 0)

# --- 4. Calculate Directional Variograms ---
n_lags_1d = 15  # Number of lag bins
max_lag_1d = np.sqrt(max_lag_x**2 + max_lag_y**2)
lag_bins_1d = np.linspace(0, max_lag_1d, n_lags_1d + 1)
lag_centers_1d = (lag_bins_1d[:-1] + lag_bins_1d[1:]) / 2

directions = {
    'Omnidirectional': {'sum_sq_diff': np.zeros(n_lags_1d), 'count': np.zeros(n_lags_1d)},
    '0-180 (E-W)': {'sum_sq_diff': np.zeros(n_lags_1d), 'count': np.zeros(n_lags_1d)},
    '90-270 (N-S)': {'sum_sq_diff': np.zeros(n_lags_1d), 'count': np.zeros(n_lags_1d)},
    '45-225 (NE-SW)': {'sum_sq_diff': np.zeros(n_lags_1d), 'count': np.zeros(n_lags_1d)},
    '135-315 (NW-SE)': {'sum_sq_diff': np.zeros(n_lags_1d), 'count': np.zeros(n_lags_1d)}
}
angle_tolerance = 22.5

for i in range(len(df)):
    for j in range(i + 1, len(df)):
        lag_vector = coords[i] - coords[j]
        h_x, h_y = lag_vector[0], lag_vector[1]
        dist = np.sqrt(h_x**2 + h_y**2)
        bin_index = np.digitize(dist, lag_bins_1d) - 1

        if 0 <= bin_index < n_lags_1d:
            sq_diff = (values[i] - values[j])**2
            directions['Omnidirectional']['sum_sq_diff'][bin_index] += sq_diff
            directions['Omnidirectional']['count'][bin_index] += 1
            angle = np.degrees(np.arctan2(h_y, h_x)) % 180

            if abs(angle - 0) < angle_tolerance or abs(angle - 180) < angle_tolerance: key = '0-180 (E-W)'
            elif abs(angle - 90) < angle_tolerance: key = '90-270 (N-S)'
            elif abs(angle - 45) < angle_tolerance: key = '45-225 (NE-SW)'
            elif abs(angle - 135) < angle_tolerance: key = '135-315 (NW-SE)'
            else: key = None

            if key:
                directions[key]['sum_sq_diff'][bin_index] += sq_diff
                directions[key]['count'][bin_index] += 1

for key in directions:
    count = directions[key]['count']
    directions[key]['variogram'] = np.divide(directions[key]['sum_sq_diff'], count, out=np.full(n_lags_1d, np.nan), where=count > 0)

# --- 5. Plot the Variograms using Plotly ---
# Create a figure with two subplots: one for the 3D surface, one for the 2D lines
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'surface'}, {'type': 'xy'}]],
    subplot_titles=('3D Sample Variogram', 'Directional Variograms')
)

# Add 3D Surface plot
lag_x_centers = (lag_x_bins[:-1] + lag_x_bins[1:]) / 2
lag_y_centers = (lag_y_bins[:-1] + lag_y_bins[1:]) / 2
fig.add_trace(
    go.Surface(x=lag_x_centers, y=lag_y_centers, z=sample_variogram_2d.T, colorscale='Viridis', colorbar_title='Variogram'),
    row=1, col=1
)

# Add 2D Directional plots
colors = {'Omnidirectional': 'black', '0-180 (E-W)': 'red', '90-270 (N-S)': 'blue', '45-225 (NE-SW)': 'green', '135-315 (NW-SE)': 'purple'}
for key in directions:
    fig.add_trace(
        go.Scatter(
            x=lag_centers_1d,
            y=directions[key]['variogram'],
            mode='lines+markers',
            name=key,
            line=dict(color=colors[key], width=4 if key == 'Omnidirectional' else 2),
            marker=dict(symbol='circle' if key == 'Omnidirectional' else 'x')
        ),
        row=1, col=2
    )


fig.show()
